見つからない

In [1]:
import random
import numpy as np
import sys
from scipy.sparse import csr_matrix, hstack, vstack

# --- 表示設定 ---
# 行列の全要素を表示し、省略されないように設定する。
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

class HighPerformanceGirthOptimizer:
    """
    グラフ探索に基づき、ガース（閉路長）を最大化しながら
    量子LDPC符号のパリティ検査行列を構成する。
    """
    def __init__(self, P=768, J=3, L_half=6):
        self.P = P
        self.J = J
        self.L_half = L_half
        self.mid = P // 2
        # 基底（歯車）の生成
        self.rho_A = self._generate_random_cycle(range(0, self.mid))
        self.rho_B = self._generate_random_cycle(range(self.mid, self.P))
        self.identity = tuple(range(P))
        self.zero_mat = csr_matrix((P, P), dtype=np.int8)

    def _generate_random_cycle(self, r):
        indices = list(r)
        random.shuffle(indices)
        p = list(range(self.P))
        for i in range(len(indices)):
            p[indices[i]] = indices[(i + 1) % len(indices)]
        return tuple(p)

    def get_power(self, base_p, k):
        """置換の冪乗を高速に計算する。"""
        res = list(range(self.P))
        curr = list(base_p)
        while k > 0:
            if k % 2 == 1:
                res = [res[curr[i]] for i in range(self.P)]
            curr = [curr[curr[i]] for i in range(self.P)]
            k //= 2
        return tuple(res)

    def inject_swap(self, p, r):
        """特定の領域内でスワップを行い、非可換性を注入する。"""
        p_list = list(p)
        indices = list(r)
        if len(indices) < 2: return p
        i1, i2 = random.sample(indices, 2)
        p_list[i1], p_list[i2] = p_list[i2], p_list[i1]
        return tuple(p_list)

    def build_partial_hx(self, F_list, G_list):
        """決定済みのブロックのみで Hx を構成し、未決定部分は零行列とする。"""
        F_m = [tuple_to_sparse(f, self.P) for f in F_list] + [self.zero_mat] * (self.L_half - len(F_list))
        G_m = [tuple_to_sparse(g, self.P) for g in G_list] + [self.zero_mat] * (self.L_half - len(G_list))
        
        rows = []
        for i in range(self.J):
            row = [F_m[(j - i) % self.L_half] for j in range(self.L_half)] + \
                  [G_m[(j - i) % self.L_half] for j in range(self.L_half)]
            rows.append(hstack(row))
        return vstack(rows)

    def count_cycles_direct(self, H):
        """
        隣接リストを用いてタンナーグラフから直接サイクルをカウントする。
        """
        num_checks, _ = H.shape
        # 行列が空（決定前）の場合は 0 を返す
        if H.nnz == 0: return 0, 0
        
        # 4-cycle の計算 (高速な行間比較)
        H_int = H.astype(np.int64)
        Gram = (H_int @ H_int.T).toarray()
        np.fill_diagonal(Gram, 0)
        c4 = np.sum(Gram * (Gram - 1)) // 4
        
        # 6-cycle の計算 (計算コストが大きいため必要に応じて呼び出す)
        # 本探索では主に c4 == 0 を目標とする。
        return int(c4)

    def optimize(self):
        """逐次追加法により、サイクル 0 の構成を探索する。"""
        print(f"探索開始: P={self.P}, J={self.J}")
        
        while True:
            F_final = []
            G_final = []
            failed = False
            
            # F ブロックの決定
            for i in range(self.L_half):
                found = False
                candidates = random.sample(range(1, self.mid), min(100, self.mid-1))
                for k in candidates:
                    test_f = self.get_power(self.rho_A, k)
                    # 条件Bの注入 (F0)
                    if i == 0: test_f = self.inject_swap(test_f, range(0, self.mid))
                    
                    Hx = self.build_partial_hx(F_final + [test_f], G_final)
                    if self.count_cycles_direct(Hx) == 0:
                        F_final.append(test_f)
                        found = True
                        break
                if not found:
                    failed = True; break
                print(f"F[{i}] 確定")

            if failed: 
                print("F構成失敗、リスタート..."); continue

            # G ブロックの決定
            for j in range(self.L_half):
                found = False
                candidates = random.sample(range(1, self.mid), min(100, self.mid-1))
                for k in candidates:
                    test_g = self.get_power(self.rho_B, k)
                    # 条件Bの注入 (G3, G2)
                    if j == 3: 
                        test_g = [test_g[self.get_power(self.rho_A, random.randint(1, self.mid-1))[m]] for m in range(self.P)]
                    if j == 2:
                        test_g = self.inject_swap(test_g, range(self.mid, self.P))
                        
                    Hx = self.build_partial_hx(F_final, G_final + [test_g])
                    if self.count_cycles_direct(Hx) == 0:
                        G_final.append(test_g)
                        found = True
                        break
                if not found:
                    failed = True; break
                print(f"G[{j}] 確定")

            if failed: 
                print("G構成失敗、リスタート..."); continue
            
            return F_final, G_final

def tuple_to_sparse(p, size):
    rows = np.arange(size)
    return csr_matrix((np.ones(size, dtype=np.int8), (rows, p)), shape=(size, size))

def final_build(F, G, P, J):
    """最終的な Hx, Hz を組み立てる。"""
    L_h = len(F)
    F_m = [tuple_to_sparse(f, P) for f in F]
    G_m = [tuple_to_sparse(g, P) for g in G]
    hx_rows, hz_rows = [], []
    for i in range(J):
        row_x = [F_m[(j-i)%L_h] for j in range(L_h)] + [G_m[(j-i)%L_h] for j in range(L_h)]
        row_z = [G_m[(i-j)%L_h].transpose() for j in range(L_h)] + [F_m[(i-j)%L_h].transpose() for j in range(L_h)]
        hx_rows.append(hstack(row_x))
        hz_rows.append(hstack(row_z))
    return vstack(hx_rows), vstack(hz_rows)

# --- 実行セクション ---
P, J = 768, 3
opt = HighPerformanceGirthOptimizer(P=P, J=J)
F_res, G_res = opt.optimize()

Hx, Hz = final_build(F_res, G_res, P, J)

# 1. 直交性チェック
ortho = (Hx @ Hz.transpose())
ortho.data %= 2
ortho.eliminate_zeros()
print(f"\nCSS条件違反数 (nnz): {ortho.nnz}")

# 2. 最終サイクルカウント
H_int = Hx.astype(np.int64)
Gram = (H_int @ H_int.T).toarray()
np.fill_diagonal(Gram, 0)
c4_final = np.sum(Gram * (Gram - 1)) // 4
print(f"最終構成の長さ4サイクル数: {c4_final}")

探索開始: P=768, J=3
F[0] 確定
F[1] 確定
F構成失敗、リスタート...
F[0] 確定
F[1] 確定


KeyboardInterrupt: 